In [1]:
!pip install scikit-learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# FAQ data
faqs = [
    {
        "question": "How do I reset my password?",
        "answer": "Use the recovery link on the login page.",
        "source": "it_help.txt"
    },
    {
        "question": "Where do I submit my assignment?",
        "answer": "Use the assignment upload page in the learning portal.",
        "source": "assessment.txt"
    },
    {
        "question": "Can illness justify a late submission?",
        "answer": "Request an extension with medical evidence before the deadline.",
        "source": "handbook.txt"
    },
    {
        "question": "Where can I find the semester timetable?",
        "answer": "Check the university learning portal for the latest semester timetable.",
        "source": "timetable.txt"
    },
    {
        "question": "How do I contact the university IT help desk?",
        "answer": "Contact the IT support team through the university help desk.",
        "source": "it_support.txt"
    }
]


# Prepare FAQ text
faq_text = []

for faq in faqs:
    faq_text.append(faq["question"] + " " + faq["answer"])


# Apply TF-IDF
vectorizer = TfidfVectorizer()
faq_vectors = vectorizer.fit_transform(faq_text)


# Chatbot function
def chatbot_reply(text):

    if text.strip() == "":
        print("Chatbot: Please enter a question.")
        return

    # Check exact question
    for faq in faqs:
        if text.lower().strip() == faq["question"].lower():

            print("Chatbot:", faq["answer"])
            print("Source:", faq["source"])
            return

    # Search similar FAQ
    user_vector = vectorizer.transform([text])

    if user_vector.nnz == 0:
        print("Chatbot: No matching FAQ. Please contact the help desk.")
        return

    scores = cosine_similarity(user_vector, faq_vectors)[0]

    print("Chatbot: Possible matches. Choose or rephrase your question.")

    count = 0

    for i in scores.argsort()[::-1]:

        print(faqs[i]["question"], "Score:", round(scores[i], 3))

        count += 1

        if count == 3:
            break


# Main chatbot
def chatbot_app():

    print("--- Welcome to University Information Chatbot ---")
    print("--- Type 'exit' to stop ---")

    while True:

        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Chatbot: Goodbye!")
            break

        else:
            chatbot_reply(user_input)


chatbot_app()

--- Welcome to FAQ Chatbot ---
--- Type 'exit' to stop ---
You: How do I reset my password?
Chatbot: Use the recovery link on the login page.
Source: it_help.txt
You: Can illness justify a late submission?
Chatbot: Request an extension with medical evidence before the deadline.
Source: handbook.txt
You: Where do I submit assignment?
Chatbot: Possible matches. Choose or rephrase your question.
Where do I submit my assignment? Score: 0.665
Where can I find the semester timetable? Score: 0.089
How do I reset my password? Score: 0.081
You: exit
Chatbot: Goodbye!
